# Paracetamol: torsional states, and the symmetry that merges them

`md_tools.analysis.t_hdbscan` partitions the torsional ensemble into density basins;
`t_symmetry` then asks which of those basins are the SAME conformer seen through a permutation of
chemically identical atoms, and merges only those.

**A cluster is a density basin, not a metastable state.** Nothing here looks at time. Two peaks
separated by a sampled region are two clusters whatever the barrier between them.

**Needs the analysis environment** (`scikit-learn`, `rdkit`): see
[Installing](../../../install.md#5-optional-the-analysis-environment).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "_analysis"))   # docs/tutorial/_analysis

import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md
import tutorial_data as td

# Where the runs are. NOT defaulted: see docs/tutorial/_analysis/tutorial_data.py
print("runs root:", td.runs_root())

In [ ]:
PARA_TORSIONS = {"omega": (0, 1, 3, 4), "aryl": (1, 3, 4, 5), "phenol": (6, 7, 8, 17)}
names, quads = list(PARA_TORSIONS), list(PARA_TORSIONS.values())

In [ ]:
from md_tools.analysis import t_hdbscan

cmd = td.solute("PARA-1us", stride=25)          # 4000 frames is plenty for three torsions
tors = md.compute_dihedrals(cmd, quads)
print(f"{tors.shape[0]} frames, {tors.shape[1]} torsions")

fit = t_hdbscan(tors, names=names).fit()
summary = fit.summary()
print(f"n_clusters = {summary['n_clusters']}")
print(f"masses     = { {k: round(v, 3) for k, v in summary['cluster_mass'].items()} }")
print(f"noise      = {summary['noise_mass']:.4f}")

## Is one state a possible answer here?

`allow_single_cluster` defaults to **True** in md-tools, which is a deliberate departure from
sklearn's default. With it False, HDBSCAN's EOM selection refuses the root of the condensed tree,
so a UNIMODAL ensemble cannot come back as one state -- it is split into two or three with a
plausible mass split and nothing in the output saying one state was never a candidate.

Paracetamol is a small, rigid molecule, so it is exactly the kind of system where that would have
bitten. Check it both ways rather than trusting either default.

In [ ]:
for flag in (True, False):
    s = t_hdbscan(tors, names=names, allow_single_cluster=flag).fit().summary()
    print(f"allow_single_cluster={flag!s:5s} -> {s['n_clusters']} clusters, "
          f"excluded_by_construction={s['single_cluster_excluded_by_construction']}")
for caveat in summary["caveats"]:
    print("-", caveat)

The answer is the same either way here, so the structure is real rather than an artefact of the
default. Had it differed, the number to believe would be the one from `True` -- and the marginals
in the PMF notebook would settle it.

## Which basins are the same conformer

The four clusters are two torsions each taking two values. But the phenyl ring has a two-fold
axis: flipping it exchanges chemically identical atoms and changes no physics. `t_symmetry` finds
that operation from the molecular graph and tests whether it maps one cluster's JOINT
distribution onto another's.

In [ ]:
from rdkit import Chem
from md_tools.analysis import load_torsion_json, load_cluster_torsion, TorsionSymmetry
import json, tempfile, os

mol = Chem.SDMolSupplier(str(td.run_dir("PARA-REST2") / "build/TYL.sdf"), removeHs=False)[0]

# The SDF's atom order and the trajectory's are NOT assumed equal -- verified by bond graph.
from md_tools.analysis import verify_atom_mapping
bonds = [(b[0].index, b[1].index) for b in cmd.topology.bonds]
elements = [a.element.symbol for a in cmd.topology.atoms]
mapping, why = verify_atom_mapping(mol, bonds, elements)
print("atom mapping:", "identity" if mapping == list(range(mol.GetNumAtoms())) else mapping)
print(" ", why)

In [ ]:
definition = {"schema_version": 1, "indexing": "zero-based", "units": "radian",
              "torsions": [{"name": n, "atoms": list(q)} for n, q in PARA_TORSIONS.items()]}
path = pathlib.Path(tempfile.mkdtemp()) / "torsion_cv.json"
path.write_text(json.dumps(definition, indent=2))

torsion_idx = load_torsion_json(path, molecule=mol)
clusters = load_cluster_torsion(tors, fit.labels_, noise_label=-1, angle_unit="radian",
                                names=list(torsion_idx.names))
print("cluster labels:", clusters.cluster_labels, "| masses:", clusters.mass_table())

In [ ]:
detector = TorsionSymmetry(mol, torsion_idx)
detector.fit(clusters, coordinates=cmd.xyz, atom_mapping=mapping)

print("operations distinct on these torsions:", len(detector.operations))
print("groups:", detector.cluster_groups)
for c in detector.comparisons:
    if c.evidence.get("role") == "transitivity-guard":
        continue
    print(f"  {c.a} vs {c.b}: {c.verdict:9s} op={c.operation} "
          f"E={c.energy if c.energy is None else round(c.energy, 4)}")

**Not all four merge.** The ring flip relates 0 to 2 and 1 to 3, but nothing relates those two
groups: one is the pairing where both torsions sit together, the other where they are opposite,
and those are genuinely different relative orientations. The tolerance is calibrated from the
within-cluster split-half variability of THESE data, not from a universal constant.

In [ ]:
detector.draw_operation_effect(group=0, output="operation.png")
detector.draw_sym_group(0, output="group_0.png", cluster_torsions=clusters)
detector.draw_sym_group(1, output="group_1.png", cluster_torsions=clusters)

from IPython.display import Image, display
for name in ("operation.png", "group_0.png", "group_1.png"):
    display(Image(filename=name))

In [ ]:
merged = detector.deduplicate(clusters)
print(json.dumps(merged.describe(), indent=2))
assert abs(sum(merged.masses.values()) + merged.noise_mass - 1.0) < 1e-9
print("\npopulations are conserved: merged masses + noise == 1 exactly")

Masses SUM across a merge; nothing is multiplied by the symmetry order. The group's order says
how many labels the ensemble was split into, not how much probability it holds -- and the noise
label keeps its own mass rather than being absorbed.

**A structural claim is not a thermodynamic one.** That two clusters are related by a graph
automorphism says their conformations are equivalent. If an atom-specific restraint or a REST2
region covers one side and not the other, the symmetry is broken even though the graph is
unchanged -- pass `broken_by=[...]` to record that, and it travels with every accepted pair.